# Copa 2026 — ingestão de dados Kaggle, base combinada e simulação

Este notebook recria a pasta `Data/`, carrega os datasets Kaggle solicitados, combina as fontes por seleção e gera:

- base processada com 48 seleções;
- tabelas de fase de grupos simulada;
- 8 melhores terceiros;
- chaveamento resolvido a partir do arquivo de slots da Copa 2026;
- arquivo Excel com guias separadas.

A simulação é probabilística e usa seed fixa (`SEED = 42`) para reprodutibilidade.

In [1]:
from pathlib import Path
import json
import pandas as pd

SEED = 42
ROOT = Path.cwd()
DATA = ROOT / "Data"
PROCESSED = DATA / "processed"
print("Projeto:", ROOT)
print("Pasta Data:", DATA)

Projeto: /home/gabriel/Projetos /Vencedor_copa
Pasta Data: /home/gabriel/Projetos /Vencedor_copa/Data


## 1. Fontes baixadas do Kaggle

In [2]:
manifest = json.loads((DATA / "dataset_manifest.json").read_text(encoding="utf-8"))
fontes = pd.DataFrame([
    {"label": item.get("label"), "dataset": item.get("dataset"), "local_path": item.get("local_path"), "arquivos": len(item.get("files", []))}
    for item in manifest
])
display(fontes)

,label,dataset,local_path,arquivos
0,fc26,justdhia/ea-sports-fc-26-player-ratings,/home/gabriel/Projetos /Vencedor_copa/Data/raw...,3
1,elo,afonsofernandescruz/2026-fifa-world-cup-histor...,/home/gabriel/Projetos /Vencedor_copa/Data/raw...,2
2,fc25,samandarabdujabbar/ea-sports-fc-25-complete-pl...,/home/gabriel/Projetos /Vencedor_copa/Data/raw...,1
3,wc2026,pranishkessi/fifa-world-cup-2026-prediction-si...,/home/gabriel/Projetos /Vencedor_copa/Data/raw...,15


## 2. Amostras dos datasets brutos

In [3]:
raw_files = {
    "EA FC 26 jogadores": DATA / "raw/kaggle/justdhia__ea-sports-fc-26-player-ratings/ea_fc26_players.csv",
    "Elo histórico": DATA / "raw/kaggle/afonsofernandescruz__2026-fifa-world-cup-historical-elo-ratings/elo_ratings_wc2026.csv",
    "EA FC 25 stats": DATA / "raw/kaggle/samandarabdujabbar__ea-sports-fc-25-complete-player-stats-and-analysis/ea_sports_fc25_full.csv",
    "Grupos 2026": DATA / "raw/kaggle/pranishkessi__fifa-world-cup-2026-prediction-simulator/data/worldcup_2026/worldcup_2026_groups.csv",
    "Probabilidades jogos de grupo": DATA / "raw/kaggle/pranishkessi__fifa-world-cup-2026-prediction-simulator/data/worldcup_2026/model_v5_group_match_probabilities.csv",
    "Slots do chaveamento": DATA / "raw/kaggle/pranishkessi__fifa-world-cup-2026-prediction-simulator/data/worldcup_2026/worldcup_2026_knockout_slots.csv",
}
for nome, path in raw_files.items():
    df = pd.read_csv(path, nrows=5)
    print()
    print(f"=== {nome} ===")
    print(path)
    display(df)


=== EA FC 26 jogadores ===
/home/gabriel/Projetos /Vencedor_copa/Data/raw/kaggle/justdhia__ea-sports-fc-26-player-ratings/ea_fc26_players.csv


,id,rank,overallRating,firstName,lastName,commonName,birthdate,height,weight,skillMoves,...,headingAccuracy,aggression,jumping,stamina,strength,gkDiving,gkHandling,gkKicking,gkPositioning,gkReflexes
0,209331,1,91,Mohamed,Salah,NaN,6/15/1992 12:00:00 AM,175,72,4,...,59,63,79,88,75,14,14,9,11,14
1,231747,3,91,Kylian,Mbappé,NaN,12/20/1998 12:00:00 AM,182,75,5,...,78,61,90,83,77,13,5,7,11,6
2,231443,5,90,Ousmane,Dembélé,NaN,5/15/1997 12:00:00 AM,178,67,5,...,74,58,84,76,69,6,6,14,10,13
3,231866,6,90,Rodrigo,Hernández Cascante,Rodri,6/22/1996 12:00:00 AM,190,82,3,...,81,85,83,91,83,10,10,7,14,8
4,203376,8,90,Virgil,van Dijk,NaN,7/8/1991 12:00:00 AM,193,92,2,...,88,85,89,75,93,13,10,13,11,11



=== Elo histórico ===
/home/gabriel/Projetos /Vencedor_copa/Data/raw/kaggle/afonsofernandescruz__2026-fifa-world-cup-historical-elo-ratings/elo_ratings_wc2026.csv


,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0



=== EA FC 25 stats ===
/home/gabriel/Projetos /Vencedor_copa/Data/raw/kaggle/samandarabdujabbar__ea-sports-fc-25-complete-player-stats-and-analysis/ea_sports_fc25_full.csv


...

,group,team_slot,team,is_host_team,host_country_if_host,first_match_id
0,A,A1,South Africa,False,NaN,1
1,A,A2,Mexico,True,Mexico,1
2,A,A3,Czech Republic,False,NaN,2
3,A,A4,South Korea,False,NaN,2
4,B,B1,Bosnia and Herzegovina,False,NaN,3



=== Probabilidades jogos de grupo ===
/home/gabriel/Projetos /Vencedor_copa/Data/raw/kaggle/pranishkessi__fifa-world-cup-2026-prediction-simulator/data/worldcup_2026/model_v5_group_match_probabilities.csv


,match_id,stage,group,match_date,stadium,city,host_country,team_a,team_b,p_team_a_win,p_draw,p_team_b_win,most_likely_result
0,1,Group stage,A,2026-06-11,Estadio Azteca,Mexico City,Mexico,Mexico,South Africa,0.514984,0.357512,0.127504,team_a_win
1,2,Group stage,A,2026-06-12,Estadio Akron,Guadalajara,Mexico,South Korea,Czech Republic,0.347715,0.380404,0.271881,draw
2,3,Group stage,B,2026-06-12,BMO Field,Toronto,Canada,Canada,Bosnia and Herzegovina,0.539192,0.318991,0.141816,team_a_win
3,4,Group stage,D,2026-06-13,SoFi Stadium,Los Angeles,USA,USA,Paraguay,0.290801,0.356881,0.352318,draw
4,5,Group stage,D,2026-06-13,BC Place,Vancouver,Canada,Australia,Turkey,0.320830,0.319911,0.359259,team_b_win



=== Slots do chaveamento ===
/home/gabriel/Projetos /Vencedor_copa/Data/raw/kaggle/pranishkessi__fifa-world-cup-2026-prediction-simulator/data/worldcup_2026/worldcup_2026_knockout_slots.csv


,match_id,stage,stage_order,round_multiplier,date_utc,match_date,venue,stadium,city,host_country,...,away_slot_type,away_group_ref,away_match_ref,away_allowed_third_groups,predicted_home_team,predicted_away_team,is_knockout,requires_slot_resolution,source_file,source_row_index
0,73,Round of 32,2,1,2026-06-28T19:00:00Z,2026-06-28,"SoFi Stadium, Los Angeles",SoFi Stadium,Los Angeles,USA,...,runner_up_group,B,NaN,NaN,NaN,NaN,True,True,datalab_export_2026-06-03 16_17_37.xlsx,0
1,74,Round of 32,2,1,2026-06-29T17:00:00Z,2026-06-29,"NRG Stadium, Houston",NRG Stadium,Houston,USA,...,runner_up_group,F,NaN,NaN,NaN,NaN,True,True,datalab_export_2026-06-03 16_17_37.xlsx,1
2,75,Round of 32,2,1,2026-06-29T20:30:00Z,2026-06-29,"Gillette Stadium, Boston",Gillette Stadium,Boston,USA,...,best_third,NaN,NaN,"A,B,C,D,F",NaN,NaN,True,True,datalab_export_2026-06-03 16_17_37.xlsx,2
3,76,Round of 32,2,1,2026-06-30T01:00:00Z,2026-06-30,"Estadio BBVA, Monterrey",Estadio BBVA,Monterrey,Mexico,...,runner_up_group,C,NaN,NaN,NaN,NaN,True,True,datalab_export_2026-06-03 16_17_37.xlsx,3
4,77,Round of 32,2,1,2026-06-30T17:00:00Z,2026-06-30,"AT&T Stadium, Dallas",AT&T Stadium,Dallas,USA,...,runner_up_group,I,NaN,NaN,NaN,NaN,True,True,datalab_export_2026-06-03 16_17_37.xlsx,4


## 3. Base combinada gerada

In [4]:
base = pd.read_csv(PROCESSED / "copa_2026_master_team_dataset.csv")
print("Dimensão da base combinada:", base.shape)
print("Seleções:", base["team"].nunique())
print("Grupos:", base["group"].nunique())
display(base[[
    "group", "team_slot", "team", "combined_rank", "combined_strength",
    "champion_probability", "elo_rating", "elo_rank",
    "fc26_top23_overall_mean", "fc25_top23_overall_mean", "fc25_total_value_eur_top23"
]].sort_values("combined_rank").head(20))

Dimensão da base combinada: (48, 66)
Seleções: 48
Grupos: 12


,group,team_slot,team,combined_rank,combined_strength,champion_probability,elo_rating,elo_rank,fc26_top23_overall_mean,fc25_top23_overall_mean,fc25_total_value_eur_top23
0,H,H2,Spain,1,0.956654,0.2909,2165.0,1.0,84.956522,84.086957,1.121703e+09
1,I,I2,France,2,0.773324,0.0790,2081.0,3.0,85.173913,85.391304,1.483893e+09
2,J,J4,Argentina,3,0.724788,0.1182,2113.0,2.0,82.652174,82.000000,8.955692e+08
3,L,L2,England,4,0.667179,0.0938,2020.0,4.0,83.956522,81.173913,1.021669e+09
4,K,K2,Portugal,5,0.667045,0.0648,1984.0,5.0,83.217391,83.478261,9.190107e+08
5,C,C2,Brazil,6,0.649883,0.0521,1984.0,5.0,83.913043,82.782609,1.087562e+09
6,E,E2,Germany,7,0.641853,0.0131,1923.0,11.0,83.826087,85.391304,1.530320e+09
7,K,K3,Colombia,8,0.576778,0.0507,1975.0,7.0,77.956522,82.391304,8.565897e+08
8,L,L1,Croatia,9,0.571200,0.0250,1930.0,10.0,78.826087,83.652174,1.169230e+09
9,I,I3,Norway,10,0.554959,0.0116,1912.0,12.0,77.260870,84.565217,1.324287e+09


## 4. Validação do formato: 48 seleções, 12 grupos, 4 por grupo

In [5]:
validacao_grupos = base.groupby("group").agg(selecoes=("team", "nunique"), times=("team", lambda s: ", ".join(s))).reset_index()
display(validacao_grupos)
assert base["team"].nunique() == 48
assert base["group"].nunique() == 12
assert validacao_grupos["selecoes"].eq(4).all()
print("Validação OK: 48 seleções, 12 grupos e 4 seleções por grupo.")

,group,selecoes,times
0,A,4,"South Korea, Czech Republic, Mexico, South Africa"
1,B,4,"Switzerland, Canada, Qatar, Bosnia and Herzego..."
2,C,4,"Brazil, Morocco, Scotland, Haiti"
3,D,4,"Turkey, USA, Paraguay, Australia"
4,E,4,"Germany, Ecuador, Côte d'Ivoire, Curaçao"
5,F,4,"Japan, Netherlands, Sweden, Tunisia"
6,G,4,"Belgium, Egypt, Iran, New Zealand"
7,H,4,"Spain, Uruguay, Cabo Verde, Saudi Arabia"
8,I,4,"France, Norway, Senegal, Iraq"
9,J,4,"Argentina, Austria, Algeria, Jordan"


Validação OK: 48 seleções, 12 grupos e 4 seleções por grupo.


## 5. Resultados simulados da fase de grupos

In [6]:
group_matches = pd.read_csv(PROCESSED / "copa_2026_group_stage_simulated_matches.csv")
group_tables = pd.read_csv(PROCESSED / "copa_2026_group_stage_tables.csv")
print("Jogos de grupo simulados:", len(group_matches))
display(group_matches[["match_id", "group", "team_a", "team_b", "p_team_a_win", "p_draw", "p_team_b_win", "sim_goals_team_a", "sim_goals_team_b", "sim_winner"]].head(20))

for group in sorted(group_tables["group"].unique()):
    print()
    print(f"Grupo {group}")
    display(group_tables[group_tables["group"] == group][["group_position", "team", "played", "wins", "draws", "losses", "goals_for", "goals_against", "goal_difference", "points"]])

Jogos de grupo simulados: 72


,match_id,group,team_a,team_b,p_team_a_win,p_draw,p_team_b_win,sim_goals_team_a,sim_goals_team_b,sim_winner
0,1,A,Mexico,South Africa,0.514984,0.357512,0.127504,0,0,Draw
1,2,A,South Korea,Czech Republic,0.347715,0.380404,0.271881,1,0,South Korea
2,3,B,Canada,Bosnia and Herzegovina,0.539192,0.318991,0.141816,2,1,Canada
3,4,D,USA,Paraguay,0.290801,0.356881,0.352318,1,0,USA
4,5,D,Australia,Turkey,0.320830,0.319911,0.359259,2,3,Turkey
5,6,B,Qatar,Switzerland,0.197121,0.211177,0.591702,0,1,Switzerland
6,7,C,Brazil,Morocco,0.383771,0.362552,0.253677,1,2,Morocco
7,8,C,Haiti,Scotland,0.165988,0.309564,0.524448,2,1,Haiti
8,9,E,Germany,Curaçao,0.579904,0.316379,0.103717,2,1,Germany
9,10,F,Netherlands,Japan,0.398630,0.379426,0.221944,2,1,Netherlands



Grupo A


,group_position,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
0,1,South Korea,3,2,0,1,4,3,1,6
1,2,Mexico,3,1,2,0,3,2,1,5
2,3,South Africa,3,1,1,1,4,4,0,4
3,4,Czech Republic,3,0,1,2,3,5,-2,1



Grupo B


...

,group_position,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
36,1,Argentina,3,2,1,0,6,3,3,7
37,2,Austria,3,1,1,1,4,4,0,4
38,3,Algeria,3,1,1,1,4,5,-1,4
39,4,Jordan,3,0,1,2,3,5,-2,1



Grupo K


,group_position,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
40,1,Portugal,3,1,2,0,6,5,1,5
41,2,Colombia,3,1,2,0,6,5,1,5
42,3,DR Congo,3,0,2,1,3,4,-1,2
43,4,Uzbekistan,3,0,2,1,3,4,-1,2



Grupo L


,group_position,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
44,1,England,3,1,2,0,4,3,1,5
45,2,Panama,3,1,2,0,2,1,1,5
46,3,Croatia,3,1,1,1,4,3,1,4
47,4,Ghana,3,0,1,2,2,5,-3,1


## 6. Classificados: 2 primeiros + 8 melhores terceiros

In [7]:
best_thirds = pd.read_csv(PROCESSED / "copa_2026_best_thirds.csv")
direct = group_tables[group_tables["group_position"] <= 2].copy()
qualified = pd.concat([direct, best_thirds], ignore_index=True)
print("Classificados diretos:", len(direct))
print("Melhores terceiros:", len(best_thirds))
print("Total para Round of 32:", len(qualified))
display(best_thirds[["group", "group_position", "team", "points", "goal_difference", "goals_for", "combined_strength"]])
assert len(qualified) == 32

Classificados diretos: 24
Melhores terceiros: 8
Total para Round of 32: 32


,group,group_position,team,points,goal_difference,goals_for,combined_strength
0,L,3,Croatia,4,1,4,0.571200
1,A,3,South Africa,4,0,4,0.103553
2,E,3,Côte d'Ivoire,4,0,3,0.400729
3,F,3,Sweden,4,-1,5,0.416056
4,J,3,Algeria,4,-1,4,0.462215
5,G,3,Egypt,3,-1,6,0.415458
6,D,3,Turkey,3,-1,4,0.492838
7,C,3,Brazil,2,-1,5,0.649883


## 7. Chaveamento 2026 resolvido

In [8]:
bracket = pd.read_csv(PROCESSED / "copa_2026_resolved_bracket_simulation.csv")
display(bracket[["match_id", "stage", "slot_home", "slot_away", "team_home", "team_away", "p_home_win", "winner"]])
campeao = bracket.loc[bracket["stage"].eq("Final"), "winner"].iloc[0]
print("Campeão simulado nesta seed:", campeao)

,match_id,stage,slot_home,slot_away,team_home,team_away,p_home_win,winner
0,73,Round of 32,Runner-up Group A,Runner-up Group B,Mexico,Switzerland,0.331547,Switzerland
1,74,Round of 32,Winner Group C,Runner-up Group F,Morocco,Japan,0.452451,Japan
2,75,Round of 32,Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,South Africa,0.918516,South Africa
3,76,Round of 32,Winner Group F,Runner-up Group C,Netherlands,Haiti,0.870549,Netherlands
4,77,Round of 32,Runner-up Group E,Runner-up Group I,Ecuador,Norway,0.455788,Norway
5,78,Round of 32,Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Sweden,0.833093,France
6,79,Round of 32,Winner Group A,Best 3rd (Groups C/E/F/H/I),South Korea,Côte d'Ivoire,0.565699,Côte d'Ivoire
7,80,Round of 32,Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Algeria,0.715519,England
8,81,Round of 32,Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Croatia,0.462195,Belgium
9,82,Round of 32,Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Egypt,0.550566,Egypt


Campeão simulado nesta seed: Spain


## 8. Arquivos salvos

In [9]:
arquivos = [
    PROCESSED / "copa_2026_master_team_dataset.csv",
    PROCESSED / "copa_2026_group_stage_simulated_matches.csv",
    PROCESSED / "copa_2026_group_stage_tables.csv",
    PROCESSED / "copa_2026_best_thirds.csv",
    PROCESSED / "copa_2026_resolved_bracket_simulation.csv",
    PROCESSED / "copa_2026_base_combinada.xlsx",
    PROCESSED / "simulation_summary.json",
]
for path in arquivos:
    print(path.resolve(), "->", path.stat().st_size, "bytes")

/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_master_team_dataset.csv -> 28718 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_group_stage_simulated_matches.csv -> 11870 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_group_stage_tables.csv -> 2454 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_best_thirds.csv -> 510 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_resolved_bracket_simulation.csv -> 4918 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/copa_2026_base_combinada.xlsx -> 43256 bytes
/home/gabriel/Projetos /Vencedor_copa/Data/processed/simulation_summary.json -> 202 bytes
